# Recommendation 02 — Baseline e selezione sulla validation

Il riferimento principale è **B, k=10**. I film senza giudizio danno zero guadagno osservato e non sono classificati come dislike. Le curve a k diversi hanno target B diversi. Il test già ispezionato rimane esplorativo. Ogni report indica modalità e sottoinsieme; le repliche sono mediate per utente prima del bootstrap.

Le celle sono intenzionalmente disattivate: attivare soltanto la fase desiderata. Nessun risultato precedente è sovrascritto. Vedere `docs/experiments/recommendation-coldstart-v2.md`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p/'configs/recommendation_coldstart_v2.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Aprire il notebook dalla repository o da notebooks/')
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
from movie_rag.recommendation_benchmark import BenchmarkConfig, Protocol
CONFIG = BenchmarkConfig.load(ROOT)
PROTOCOL_ID = CONFIG.values['protocol_id']
print('Protocollo:', PROTOCOL_ID, '| configurazione:', CONFIG.fingerprint[:12])


Sistemi: `bayesian_mean`, `most_popular`, MF completa, MF solo bias, GTE puro e GTE+prior. Il pilot esplora un sottoinsieme; **solo full_validation congela i parametri**. Il test richiede quella selezione e gli embedding v2. Tutti i candidati sono generati senza accedere ai giudizi nascosti.

In [ ]:
RUN_MODE = 'pilot_validation'  # pilot_validation, full_validation, test
from movie_rag.recommendation_experiments import run_baselines
RUN_BASELINES = False
if RUN_BASELINES:
    protocol = Protocol.load(CONFIG)
    report = run_baselines(protocol, RUN_MODE)
    display(report['macro'].query("scheme == 'B' and k == 10"))
    display(report['paired'].query("scheme == 'B' and k == 10"))
    print('Report:', report['directory'])
else:
    print('Training e valutazione disattivati. Modalità:', RUN_MODE)
